In [8]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [9]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AIzaSyDs


In [10]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

initialise our model here

In [11]:
MODEL = 'gpt-4.1-mini'

In [12]:
def pick_agent(user_message, history):
    messages = [
        {
            "role": "system",
            "content":
                "You are a routing classifier for feature requests. "
                "Decide which agent should answer: 'leave', 'devices', 'claims' or 'general'. "
                "Leave: Involves querying leave balance, drafting and submitting leave requests such as medical, annual and family. "
                "Devices: Involves requesting for new device accessories such as cables and peripherals. "
                "Claims: Involves medical reimbursement claims based on receipts. "
                "General: Other unrelated queries or when user asks what you can do."
        },
    ]

    if history:
        messages.extend(history)
    
    messages.append({"role": "user", "content": user_message})

    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    intent = response.choices[0].message.content.lower()
    if "leave" in intent:
        return "leave"
    if "device" in intent:
        return "devices"
    if "claim" in intent:
        return "claims"
    return "general"

In [13]:
from src.agents.smol_leave_agent import leave_smol_agent
from src.agents.smol_device_agent import device_smol_agent
from src.agents.smol_common_agent import common_smol_agent
from src.agents.smol_claims_agent import claims_smol_agent

def build_context(message, history):
    if not history:
        return message
    parts = [f"{h['role']}: {h['content']}" for h in history]
    parts.append(f"user: {message}")
    return "\n".join(parts)

def unified_chat(message, history):
    agent_name = pick_agent(message, history)
    print(f"--- SYSTEM: Request routed to {agent_name} agent")

    task = build_context(message, history)

    if agent_name == "leave":
        return leave_smol_agent.run(task)
    elif agent_name == "devices":
        return device_smol_agent.run(task)
    elif agent_name == "claims":
        return claims_smol_agent.run(task)
    else:
        return common_smol_agent.run(task)

In [ ]:
import gradio as gr

view = gr.ChatInterface(
    fn=unified_chat,
    title="SimpliAsk HR Agent",
    description="Check your terminal to see the Agent Traces (routing + tools) in real-time.",
    examples=[
        "I would like to apply for 7 days of annual leave.",
        "I would like to request for HDMI Cable.",
        "Help me submit a medical claim from my receipt.",
        "List workflows you can assist me with.",
    ],
)

view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://63b9ea32d9dafac963.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


--- SYSTEM: Request routed to leave agent


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ what is my leave balance                                                                                        │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4.1-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'get_leave_balance_tool' with arguments: {'employee_id': 'mark_tan', 'leave_type': 'annual'}      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'You have started a medical leave request from          │
│ 2025-12-04 to 2025-12-04. Please confirm if you want to proceed with submitting this request or provide an end  │
│ date if the leave is longer.'}                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: You have started a medical leave request from 2025-12-04 to 2025-12-04. Please confirm if you want to
proceed with submitting this request or provide an end date if the leave is longer.

Final answer: You have started a medical leave request from 2025-12-04 to 2025-12-04. Please confirm if you want to
proceed with submitting this request or provide an end date if the leave is longer.

[Step 3: Duration 1.93 seconds| Input tokens: 6,938 | Output tokens: 128]